In [1]:
from bs4 import BeautifulSoup
import requests
import time
import random
import csv
import re
from urllib.parse import urljoin
from datetime import datetime

In [2]:
# Create a session object
session = requests.Session()

# Define the retry function using session
def fetch_ready_page(url, session, headers, max_retries=5, delay=5):
    for attempt in range(max_retries):
        response = session.get(url, headers=headers)
        if response.status_code == 200:
            return response
        print(f"Attempt {attempt + 1}: Status {response.status_code}, retrying in {delay} seconds...")
        time.sleep(delay)
    print(f"Failed to fetch {url} after {max_retries} attempts.")
    return None

# Load URLs from input.txt
with open("input.txt", "r") as file:
    urls = [line.strip() for line in file if line.strip()]

In [3]:
# Set up headers
# List of User-Agents to rotate
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36",
    "Mozilla/5.0 (Windows NT 6.1; WOW64; rv:40.0) Gecko/20100101 Firefox/40.0"
]

# Randomly choose a User-Agent for each request
headers = {
    "User-Agent": random.choice(user_agents)
}

In [4]:
#Define rows
rows = []

#Loop
for url in urls:
    
    base_url = "https://www.courtlistener.com"

    response = fetch_ready_page(url, session, headers, delay=2)
    if response is None:
        print(f"Skipping URL due to repeated failed responses: {url}")
        continue  # or exit, depending on how you're iterating over URLs
    html = response.text

    print(f"Fetched {url} - Status code: {response.status_code}")
    soup = BeautifulSoup(response.text, "html.parser")

    # Title
    full_title = soup.title.string.strip() if soup.title else "N/A"
    title = full_title.split(",")[0]

    # Find all entry divs
    entry_divs = soup.find_all("div", id=lambda x: x and x.startswith("entry-"))

    latest_date = "N/A"
    latest_link = "N/A"

    if entry_divs:
        latest_entry = entry_divs[-1]

        # Extract and convert date to ISO format
        entry_text = latest_entry.get_text(separator=" ", strip=True)
        date_match = re.search(r"[A-Z][a-z]{2} \d{1,2}, \d{4}", entry_text)
        if date_match:
            raw_date = date_match.group(0)
            try:
                parsed_date = datetime.strptime(raw_date, "%b %d, %Y")
                latest_date = parsed_date.strftime("%Y-%m-%d")
            except ValueError:
                latest_date = raw_date

        # First link in latest entry
        a_tag = latest_entry.find("a", href=True)
        if a_tag:
            latest_link = urljoin(base_url, a_tag["href"])

    # Prepare and store the row
    row = {
        "Title": title,
        "Link": url,
        "Date of latest entry": latest_date,
        "Link of latest entry": latest_link
    }
    rows.append(row)

Fetched https://www.courtlistener.com/docket/69609277/penguin-random-house-llc-v-labrador/ - Status code: 200
Fetched https://www.courtlistener.com/docket/69543248/st-lukes-health-system-ltd-v-labrador/ - Status code: 200
Attempt 1: Status 202, retrying in 2 seconds...
Attempt 2: Status 202, retrying in 2 seconds...
Attempt 3: Status 202, retrying in 2 seconds...
Attempt 4: Status 202, retrying in 2 seconds...
Attempt 5: Status 202, retrying in 2 seconds...
Failed to fetch https://www.courtlistener.com/docket/69802434/idaho-organization-of-resource-councils-v-labrador/ after 5 attempts.
Skipping URL due to repeated failed responses: https://www.courtlistener.com/docket/69802434/idaho-organization-of-resource-councils-v-labrador/


In [5]:
# Write results to output.csv
with open("output.csv", mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print("All data saved to output.csv")

All data saved to output.csv
